# Project 1 - Playground

# Task

TASK 1: PREDICT THE AGE OF A BRAIN FROM MRI FEATURES

This task is primarily concerned with regression. However, we have perturbed the original MRI features in several ways.

You will have to perform the following preprocessing steps:

1. outlier detection
2. feature selection
3. imputation of missing values

---

# EDA: Perturbation hunt (2026-10-25)

**Findings (details in the cells below):**

1. **Constant columns** — `x104, x129, x489, x530` are constant in train (already dropped by `data.py`).
2. **Duplicate groups** — 5 groups of exact affine copies (incl. sign-flips and extreme rescalings) covering 38 columns. Biggest groups: 18 and 11 columns. `x665` (scale ~1e22) is an exact scaled copy of `x668`; `x596` ~ `x493`. Relations hold in test too.
3. **Cell outliers** — 41 cells with robust |z| > 20 (train), 32 in test, concentrated in `x742` (30 cells), `x413` (5), `x115` (3 — the strongest target feature!).
4. **Missingness is MCAR** — 5.4–10% per column (no column complete), no cross-column block structure, no correlation with target (max |r| = 0.088).
5. **No pure-noise columns** — every column correlates >= 0.3 (rank) with at least one other column. Feature selection is about redundancy, not junk removal.
6. **No train/test shift** — max |mean shift| = 0.17 sd; missing rates match.
7. **Target signal is concentrated** — 11 features with |rho(y)| > 0.5, 65 > 0.3, median 0.028.

**Pipeline actions:** winsorize cells at robust |z| ~ 15–20 (x742/x115/x413 sabotage); z-scale + average duplicate groups (denoises, fixes extreme scales); KNN impute (MCAR); keep iForest row removal.

In [1]:
import numpy as np
import pandas as pd
from collections import defaultdict

X_tr = pd.read_csv("../data/X_train.csv", index_col=0)
X_te = pd.read_csv("../data/X_test.csv", index_col=0)
y_tr = pd.read_csv("../data/y_train.csv", index_col=0)["y"]
cols = [c for c in X_tr.columns if X_tr[c].nunique() > 1]  # drop constants
Xt, Xe = X_tr[cols], X_te[cols]
print(f"train {Xt.shape}, test {Xe.shape}, target std {y_tr.std():.1f}")
print(f"constant columns dropped: {sorted(set(X_tr.columns) - set(cols))}")

train (1212, 828), test (776, 828), target std 9.7
constant columns dropped: ['x104', 'x129', 'x489', 'x530']


## 1. Missingness — MCAR, uniform injection

In [2]:
M = Xt.isna()
per_col = M.mean()
print(f"missing cells: train {M.to_numpy().mean():.2%}, test {Xe.isna().to_numpy().mean():.2%}")
print(f"per-column rate: min {per_col.min():.3f}, median {per_col.median():.3f}, max {per_col.max():.3f}")
miss_y = [np.corrcoef(M[c], y_tr)[0, 1] for c in cols]
print(f"missingness-vs-target |corr|: max {np.abs(miss_y).max():.3f} (chance ~0.06) -> no MAR w.r.t. target")
Mc = M.astype(float).corr().to_numpy().copy()
np.fill_diagonal(Mc, 0)
print(f"missingness cross-corr max: {np.nanmax(Mc):.3f} -> no block structure")

missing cells: train 7.63%, test 6.29%
per-column rate: min 0.054, median 0.077, max 0.100
missingness-vs-target |corr|: max 0.088 (chance ~0.06) -> no MAR w.r.t. target


missingness cross-corr max: 0.164 -> no block structure


## 2. Scale anomalies — 3 astronomical columns, recoverable as duplicates

In [3]:
maxabs = Xt.abs().max()
print(pd.cut(maxabs, [0, 1e3, 1e6, 1e9, 1e24]).value_counts().sort_index().to_string())

Xr = Xt.rank(pct=True)
yt = y_tr.rank(pct=True)
for col in ["x665", "x173", "x596"]:
    cors = Xr.apply(lambda s: np.corrcoef(Xr[col][s.notna()], s[s.notna()])[0, 1])
    cors = cors.drop(col).abs().sort_values(ascending=False)
    rho_y = np.corrcoef(Xr[col].dropna(), yt[Xr[col].notna()])[0, 1]
    print(f"{col} (max {maxabs[col]:.1e}): best partner {cors.index[0]} |rho|={cors.iloc[0]:.3f}, |rho_y|={abs(rho_y):.3f}")

(0.0, 1000.0]                254
(1000.0, 1000000.0]          476
(1000000.0, 1000000000.0]     95
(1000000000.0, 1e+24]          3
x665 (max 9.9e+23): best partner x0 |rho|=nan, |rho_y|=0.237


x173 (max 2.9e+23): best partner x0 |rho|=nan, |rho_y|=0.113
x596 (max 8.8e+14): best partner x0 |rho|=nan, |rho_y|=0.247


## 3. Duplicate groups — exact affine copies (sign-flips included)

In [4]:
corr = Xr.corr().to_numpy().copy()
np.fill_diagonal(corr, 0)
c = np.abs(corr)
iu = np.triu_indices_from(c)
for thr in [0.999, 0.99, 0.98, 0.95, 0.9]:
    print(f"pairs with |rho| > {thr}: {(c[iu] > thr).sum()}")

# connected components at |rho| > 0.99
parent = list(range(len(cols)))
def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i
for i, j in zip(*iu):
    if c[i, j] > 0.99:
        pi, pj = find(i), find(j)
        if pi != pj:
            parent[pi] = pj
groups = defaultdict(list)
for i in range(len(cols)):
    groups[find(i)].append(cols[i])
for g in sorted(groups.values(), key=len, reverse=True):
    if len(g) > 1:
        print(f"group ({len(g)}):", g)

pairs with |rho| > 0.999: 20
pairs with |rho| > 0.99: 137
pairs with |rho| > 0.98: 201
pairs with |rho| > 0.95: 315
pairs with |rho| > 0.9: 437
group (18): ['x79', 'x123', 'x148', 'x240', 'x250', 'x316', 'x317', 'x348', 'x392', 'x442', 'x505', 'x568', 'x579', 'x614', 'x634', 'x715', 'x727', 'x790']
group (11): ['x40', 'x69', 'x151', 'x230', 'x231', 'x278', 'x288', 'x374', 'x479', 'x562', 'x675']
group (5): ['x15', 'x132', 'x610', 'x665', 'x668']
group (2): ['x202', 'x333']
group (2): ['x425', 'x774']


## 4. Cell outliers — sparse, concentrated in a few columns

In [5]:
med = Xt.median()
mad = (Xt - med).abs().median()
rz = (Xt - med).abs() / (1.4826 * mad.replace(0, np.nan))
v = rz.to_numpy().flatten()
v = v[~np.isnan(v)]
print(f"|z| > 10: {(v > 10).sum()} cells, |z| > 20: {(v > 20).sum()} cells, |z| > 50: {(v > 50).sum()}")
print(f"quantiles: 99% = {np.quantile(v, .99):.2f}, 99.9% = {np.quantile(v, .999):.2f}, max = {v.max():.0f}")
ext = (rz > 20)
print("top columns by |z|>20 count:", ext.sum().sort_values(ascending=False).head(5).to_dict())
print(f"rows affected: {ext.any(axis=1).sum()} (max 1 per row) -> row outlier removal alone cannot fix these")

|z| > 10: 167 cells, |z| > 20: 41 cells, |z| > 50: 0
quantiles: 99% = 2.81, 99.9% = 4.65, max = 49
top columns by |z|>20 count: {'x742': 30, 'x413': 5, 'x115': 3, 'x641': 2, 'x665': 1}
rows affected: 41 (max 1 per row) -> row outlier removal alone cannot fix these


## 5. Train vs test — no shift (CV should track the leaderboard)

In [6]:
shift = ((Xt.mean() - Xe.mean()).abs() / Xt.std())
print(f"|mean shift|/sd: median {shift.median():.3f}, p95 {shift.quantile(.95):.3f}, max {shift.max():.3f}")
mr = (Xt.isna().mean() - Xe.isna().mean()).abs()
print(f"missing-rate diff: median {mr.median():.4f}, max {mr.max():.4f}")

|mean shift|/sd: median 0.029, p95 0.097, max 0.173


missing-rate diff: median 0.0136, max 0.0426


## 6. Target signal — concentrated in a few dozen features

In [7]:
tgt = pd.Series(
    {col: np.corrcoef(Xr[col].dropna(), yt[Xr[col].notna()])[0, 1] for col in cols}
)
print(f"|rho(y)| > 0.5: {(tgt.abs() > .5).sum()}, > 0.3: {(tgt.abs() > .3).sum()}, > 0.2: {(tgt.abs() > .2).sum()}")
print(f"median |rho|: {tgt.abs().median():.3f}")
print(tgt.abs().sort_values(ascending=False).head(15).round(3).to_string())

|rho(y)| > 0.5: 11, > 0.3: 65, > 0.2: 138
median |rho|: 0.028
x115    0.604
x415    0.576
x159    0.551
x507    0.538
x702    0.538
x458    0.531
x485    0.526
x133    0.517
x194    0.515
x334    0.513
x465    0.500
x641    0.493
x200    0.449
x654    0.436
x712    0.434
